# 04 – Hyperparameter Optimization
Optimización con GridSearchCV, RandomizedSearchCV y Optuna.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from src.tune import (
    run_grid_search,
    run_randomized_search,
    run_optuna,
    save_tuning_results,
)

X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

C:\Users\isaia\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## GridSearchCV – Ridge
Explora exhaustivamente un grid pequeño de valores alpha.

In [2]:
grid_res = run_grid_search(X_train, y_train)
print(grid_res)


>> GridSearchCV – Ridge
  Best alpha: {'model__alpha': 100}  |  R²=0.8357

>> GridSearchCV – RandomForest (small grid)
  Best params: {'model__max_depth': None, 'model__n_estimators': 100}  |  R²=0.9188
{'Ridge': {'model__alpha': 100}, 'RandomForest_grid': {'model__max_depth': None, 'model__n_estimators': 100}}


## RandomizedSearchCV – GradientBoosting
Muestrea aleatoriamente combinaciones de hiperparámetros.

In [3]:
rand_res = run_randomized_search(X_train, y_train, n_iter=20)
print(rand_res)


>> RandomizedSearchCV – GradientBoosting (20 iterations)
  Best params: {'model__learning_rate': np.float64(0.1784569549189997), 'model__max_depth': 6, 'model__min_samples_leaf': 2, 'model__n_estimators': 301, 'model__subsample': np.float64(0.9684482051282945)}  |  R²=0.9227
{'GradientBoosting_rand': {'model__learning_rate': np.float64(0.1784569549189997), 'model__max_depth': 6, 'model__min_samples_leaf': 2, 'model__n_estimators': 301, 'model__subsample': np.float64(0.9684482051282945)}}


## Optuna (TPE) – Bayesian Optimization
Usa Tree-structured Parzen Estimators para búsqueda inteligente.

In [4]:
optuna_res = run_optuna(X_train, y_train, n_trials=30)


>> Optuna study: RandomForest_optuna (30 trials)...
  OK: Best R² = 0.8795
  >> Best params: {'n_estimators': 248, 'max_depth': 24, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt'}
  Guardado: Saved: results\plots/RandomForest_optuna_optuna_history.png
  Guardado: Saved: results\plots/RandomForest_optuna_param_importance.png

>> Optuna study: GradientBoosting_optuna (30 trials)...
  OK: Best R² = 0.9367
  >> Best params: {'n_estimators': 444, 'learning_rate': 0.04240548214218031, 'max_depth': 6, 'subsample': 0.6376336226185001, 'min_samples_leaf': 1, 'max_features': None}
  Guardado: Saved: results\plots/GradientBoosting_optuna_optuna_history.png
  Guardado: Saved: results\plots/GradientBoosting_optuna_param_importance.png


In [5]:
from IPython.display import display

df_summary = save_tuning_results(grid_res, rand_res, optuna_res)
display(df_summary)

  Guardado: Best hyperparams saved → results\metrics/best_hyperparams.json
  Guardado: Tuning summary → results\metrics/tuning_summary.csv


,Method,Model,Best R²
0,GridSearchCV,Ridge,see cv_results.csv
1,GridSearchCV,RandomForest_grid,see cv_results.csv
2,RandomizedSearchCV,GradientBoosting_rand,see cv_results.csv
3,Optuna (TPE),RandomForest_optuna,0.8795
4,Optuna (TPE),GradientBoosting_optuna,0.9367


## Análisis de importancia de hiperparámetros

Optuna permite calcular la importancia de cada hiperparámetro sobre el desempeño. Los graficos se guardaron en .

In [6]:
import matplotlib.pyplot as plt
from PIL import Image
import os

img_path = "../results/plots/RandomForest_optuna_optuna_history.png"
if os.path.exists(img_path):
    img = Image.open(img_path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Optuna Optimization History")
    plt.show()

## Análisis de Impacto: Antes vs Después de la Optimización

| Etapa | Modelo | R² CV | Método |
|-------|--------|-------|--------|
| Baseline | RandomForest (defaults) | 0.9183 | CV sin tuning |
| Grid search | RandomForest (grid) | ~0.922 | GridSearchCV |
| Random search | GradientBoosting | ~0.905 | RandomizedSearchCV |
| **Optuna TPE** | **RandomForest (Optuna)** | **0.9247** | **Bayesian opt.** |

**Conclusión:** Optuna (TPE) superó a GridSearchCV porque explora inteligentemente el espacio de hiperparámetros, priorizando regiones prometedoras en lugar de hacer una búsqueda exhaustiva ciega.

**Impacto en el rendimiento:**
- n_estimators óptimo: 342 (vs 100 default) → más árboles = menos varianza
- max_depth óptimo: 22 (vs 15 default) → mayor profundidad captura más patrones
- min_samples_leaf: 2 (vs 1 default) → reduce overfitting en hojas pequeñas


In [7]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

with open("../results/metrics/best_hyperparams.json", encoding="utf-8") as f:
    params = json.load(f)

df_tuning = pd.read_csv("../results/metrics/tuning_summary.csv")
print("Resumen de todos los metodos de optimizacion:")
print(df_tuning.to_string(index=False))

print("\nMejores hiperparametros Optuna (RandomForest):")
rf_optuna = params["RandomForest_optuna"]["best_params"]
for k, v in rf_optuna.items():
    print(f"  {k}: {v}")

for fname in [
    "RandomForest_optuna_optuna_history.png",
    "RandomForest_optuna_param_importance.png",
]:
    path = f"../results/plots/{fname}"
    if os.path.exists(path):
        img = Image.open(path)
        plt.figure(figsize=(10, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(fname.replace("_", " ").replace(".png", ""), fontweight="bold")
        plt.show()

Resumen de todos los metodos de optimizacion:
            Method                   Model            Best R²
      GridSearchCV                   Ridge see cv_results.csv
      GridSearchCV       RandomForest_grid see cv_results.csv
RandomizedSearchCV   GradientBoosting_rand see cv_results.csv
      Optuna (TPE)     RandomForest_optuna             0.9247
      Optuna (TPE) GradientBoosting_optuna             0.9108

Mejores hiperparametros Optuna (RandomForest):
  n_estimators: 342
  max_depth: 22
  min_samples_split: 4
  min_samples_leaf: 2
  max_features: sqrt
